# BirdCLEF+ 2026 — Baseline Training
**Model**: EfficientNet-B0 | **Input**: Mel Spectrogram (1×128×500)
**Task**: Multi-label sound event detection, 234 species

In [ ]:
# ── Install extra packages ────────────────────────────────────────────────────
!pip install -q timm audiomentations

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
import os, torch

class CFG:
    # Kaggle paths
    DATA_DIR    = "/kaggle/input/birdclef-2026"
    TRAIN_CSV   = f"{DATA_DIR}/train.csv"
    TAXONOMY    = f"{DATA_DIR}/taxonomy.csv"
    TRAIN_AUDIO = f"{DATA_DIR}/train_audio"
    TEST_SOUNDS = f"{DATA_DIR}/test_soundscapes"
    SUB_CSV     = f"{DATA_DIR}/sample_submission.csv"
    OUTPUT_DIR  = "/kaggle/working"

    # Audio
    SAMPLE_RATE  = 32000
    DURATION     = 5
    N_SAMPLES    = SAMPLE_RATE * DURATION

    # Mel Spectrogram
    N_FFT        = 1024
    HOP_LENGTH   = 320
    N_MELS       = 128
    FMIN         = 20
    FMAX         = 16000

    # Model
    MODEL_NAME   = "efficientnet_b0"
    NUM_CLASSES  = 234
    PRETRAINED   = True
    IN_CHANNELS  = 1

    # Training
    EPOCHS       = 20
    BATCH_SIZE   = 32       # T4 có 16GB — tăng lên 32
    NUM_WORKERS  = 2
    PIN_MEMORY   = True

    LR           = 1e-3
    WEIGHT_DECAY = 1e-4
    WARMUP_EPOCHS = 2
    MIN_LR       = 1e-6

    USE_AMP      = True
    GRAD_CLIP    = 1.0
    GRAD_ACCUM   = 2

    MIN_RATING   = 3.0

    USE_MIXUP    = True
    MIXUP_ALPHA  = 0.5
    TIME_MASK    = 20
    FREQ_MASK    = 10

    N_FOLDS      = 5
    FOLD         = 0
    SEED         = 42

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device : {CFG.DEVICE}")
if CFG.DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
import os, ast, random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
import librosa


def load_audio(path, sr=CFG.SAMPLE_RATE):
    audio, _ = librosa.load(path, sr=sr, mono=True)
    return audio.astype(np.float32)


def audio_to_melspec(audio):
    mel = librosa.feature.melspectrogram(
        y=audio, sr=CFG.SAMPLE_RATE,
        n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH,
        n_mels=CFG.N_MELS, fmin=CFG.FMIN, fmax=CFG.FMAX,
    )
    return librosa.power_to_db(mel, ref=np.max).astype(np.float32)


def pad_or_crop(audio, target_len):
    if len(audio) < target_len:
        audio = np.tile(audio, target_len // len(audio) + 1)
    start = random.randint(0, len(audio) - target_len)
    return audio[start: start + target_len]


def spec_augment(mel):
    mel = mel.copy()
    if CFG.TIME_MASK > 0:
        t  = random.randint(0, CFG.TIME_MASK)
        t0 = random.randint(0, max(0, mel.shape[1] - t))
        mel[:, t0:t0+t] = mel.min()
    if CFG.FREQ_MASK > 0:
        f  = random.randint(0, CFG.FREQ_MASK)
        f0 = random.randint(0, max(0, mel.shape[0] - f))
        mel[f0:f0+f, :] = mel.min()
    return mel


def normalize_mel(mel):
    mn, mx = mel.min(), mel.max()
    return (mel - mn) / (mx - mn) if mx > mn else mel


def build_label_map(taxonomy_csv):
    tax = pd.read_csv(taxonomy_csv)
    label2idx = {row["primary_label"]: i for i, row in tax.iterrows()}
    idx2label = {i: row["primary_label"] for i, row in tax.iterrows()}
    return label2idx, idx2label


def make_label_vector(primary, secondary, label2idx, n_classes):
    vec = np.zeros(n_classes, dtype=np.float32)
    if primary in label2idx:
        vec[label2idx[primary]] = 1.0
    for s in secondary:
        if s in label2idx:
            vec[label2idx[s]] = 0.5
    return vec


class BirdDataset(Dataset):
    def __init__(self, df, label2idx, augment=False):
        self.df        = df.reset_index(drop=True)
        self.label2idx = label2idx
        self.augment   = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(CFG.TRAIN_AUDIO, row["filename"])
        try:
            audio = load_audio(path)
        except Exception:
            audio = np.zeros(CFG.N_SAMPLES, dtype=np.float32)

        audio = pad_or_crop(audio, CFG.N_SAMPLES)
        mel   = normalize_mel(audio_to_melspec(audio))
        if self.augment:
            mel = spec_augment(mel)
        mel = torch.tensor(mel).unsqueeze(0)

        try:
            secondary = ast.literal_eval(row.get("secondary_labels", "[]"))
        except Exception:
            secondary = []

        label = torch.tensor(
            make_label_vector(row["primary_label"], secondary,
                              self.label2idx, CFG.NUM_CLASSES)
        )
        return mel, label


def mixup_batch(inputs, targets, alpha=CFG.MIXUP_ALPHA):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(inputs.size(0))
    return (lam * inputs + (1-lam) * inputs[idx],
            lam * targets + (1-lam) * targets[idx])


def get_loaders(fold=CFG.FOLD):
    df = pd.read_csv(CFG.TRAIN_CSV)
    label2idx, idx2label = build_label_map(CFG.TAXONOMY)

    if CFG.MIN_RATING is not None:
        df = df[df["rating"] >= CFG.MIN_RATING].reset_index(drop=True)
        print(f"After rating filter: {len(df)} clips")

    df["_path"] = df["filename"].apply(lambda f: os.path.join(CFG.TRAIN_AUDIO, f))
    df = df[df["_path"].apply(os.path.exists)].reset_index(drop=True)
    print(f"Files on disk: {len(df)} clips")

    skf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
    df["fold"] = -1
    for f, (_, val_idx) in enumerate(skf.split(df, df["primary_label"])):
        df.loc[val_idx, "fold"] = f

    train_df = df[df["fold"] != fold].reset_index(drop=True)
    val_df   = df[df["fold"] == fold].reset_index(drop=True)
    print(f"Fold {fold}: train={len(train_df)}, val={len(val_df)}")

    train_loader = DataLoader(
        BirdDataset(train_df, label2idx, augment=True),
        batch_size=CFG.BATCH_SIZE, shuffle=True,
        num_workers=CFG.NUM_WORKERS, pin_memory=CFG.PIN_MEMORY, drop_last=True,
    )
    val_loader = DataLoader(
        BirdDataset(val_df, label2idx, augment=False),
        batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
        num_workers=CFG.NUM_WORKERS, pin_memory=CFG.PIN_MEMORY,
    )
    return train_loader, val_loader, label2idx, idx2label

print("Dataset module OK")

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
import torch.nn as nn
import timm


class BirdCLEFModel(nn.Module):
    def __init__(self, model_name=CFG.MODEL_NAME, num_classes=CFG.NUM_CLASSES,
                 pretrained=CFG.PRETRAINED, drop_rate=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=pretrained,
            in_chans=CFG.IN_CHANNELS, num_classes=0, global_pool="avg",
        )
        dummy = torch.zeros(1, CFG.IN_CHANNELS, CFG.N_MELS, 500)
        with torch.no_grad():
            feat_dim = self.backbone(dummy).shape[-1]
        print(f"Backbone: {model_name} | Feature dim: {feat_dim}")

        self.head = nn.Sequential(
            nn.Dropout(drop_rate),
            nn.Linear(feat_dim, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(drop_rate / 2),
            nn.Linear(512, num_classes),
        )
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.head(self.backbone(x))


def compute_pos_weight(df, label2idx, n_classes, device):
    counts = np.zeros(n_classes)
    total  = len(df)
    for _, row in df.iterrows():
        if row["primary_label"] in label2idx:
            counts[label2idx[row["primary_label"]]] += 1
    pos    = np.clip(counts, 1, None)
    weight = np.clip((total - pos) / pos, 1.0, 10.0)
    return torch.tensor(weight, dtype=torch.float32).to(device)

print("Model module OK")

In [ ]:
# ── Training utilities ────────────────────────────────────────────────────────
import time
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import roc_auc_score


def seed_everything(seed=CFG.SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True


def get_scheduler(optimizer, n_warmup, n_total):
    def lr_lambda(step):
        if step < n_warmup:
            return step / max(1, n_warmup)
        progress = (step - n_warmup) / max(1, n_total - n_warmup)
        return CFG.MIN_LR/CFG.LR + (1 - CFG.MIN_LR/CFG.LR) * 0.5 * (1 + np.cos(np.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def compute_auc(targets, preds):
    scores = []
    for i in range(targets.shape[1]):
        if targets[:, i].sum() > 0:
            try:
                scores.append(roc_auc_score(targets[:, i], preds[:, i]))
            except Exception:
                pass
    return float(np.mean(scores)) if scores else 0.0


def train_epoch(model, loader, optimizer, scaler, criterion, scheduler, epoch):
    model.train()
    total_loss, steps = 0.0, 0
    optimizer.zero_grad()

    for step, (mels, labels) in enumerate(loader):
        mels   = mels.to(CFG.DEVICE, non_blocking=True)
        labels = labels.to(CFG.DEVICE, non_blocking=True)

        if CFG.USE_MIXUP and random.random() > 0.5:
            mels, labels = mixup_batch(mels, labels)

        with autocast(enabled=CFG.USE_AMP):
            loss = criterion(model(mels), labels) / CFG.GRAD_ACCUM

        scaler.scale(loss).backward()

        if (step + 1) % CFG.GRAD_ACCUM == 0:
            if CFG.GRAD_CLIP > 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        total_loss += loss.item() * CFG.GRAD_ACCUM
        steps += 1

        if step % 100 == 0:
            lr = optimizer.param_groups[0]["lr"]
            print(f"  Epoch {epoch} | Step {step}/{len(loader)} | "
                  f"Loss {total_loss/steps:.4f} | LR {lr:.2e}")

    return {"loss": total_loss / steps}


@torch.no_grad()
def val_epoch(model, loader, criterion):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []

    for mels, labels in loader:
        mels   = mels.to(CFG.DEVICE, non_blocking=True)
        labels = labels.to(CFG.DEVICE, non_blocking=True)
        with autocast(enabled=CFG.USE_AMP):
            logits = model(mels)
            total_loss += criterion(logits, labels).item()
        all_preds.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    all_preds  = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    return {
        "loss": total_loss / len(loader),
        "auc":  compute_auc(all_labels, all_preds),
        "preds": all_preds, "labels": all_labels,
    }

print("Training utilities OK")

In [ ]:
# ── Main Training Loop ────────────────────────────────────────────────────────
def train(fold=CFG.FOLD):
    seed_everything()
    os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"  BirdCLEF+ 2026 | Fold {fold} | {CFG.DEVICE}")
    print(f"{'='*60}\n")

    train_loader, val_loader, label2idx, _ = get_loaders(fold)

    model = BirdCLEFModel().to(CFG.DEVICE)

    df    = pd.read_csv(CFG.TRAIN_CSV)
    pos_w = compute_pos_weight(df, label2idx, CFG.NUM_CLASSES, CFG.DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY
    )

    total_steps  = len(train_loader) // CFG.GRAD_ACCUM * CFG.EPOCHS
    warmup_steps = len(train_loader) // CFG.GRAD_ACCUM * CFG.WARMUP_EPOCHS
    scheduler = get_scheduler(optimizer, warmup_steps, total_steps)
    scaler    = GradScaler(enabled=CFG.USE_AMP)

    best_auc, patience, no_improve = 0.0, 5, 0
    history = []

    for epoch in range(1, CFG.EPOCHS + 1):
        t0 = time.time()
        print(f"\n── Epoch {epoch}/{CFG.EPOCHS} ────────────")

        train_m = train_epoch(model, train_loader, optimizer, scaler,
                              criterion, scheduler, epoch)
        val_m   = val_epoch(model, val_loader, criterion)

        elapsed = time.time() - t0
        print(f"  Train Loss : {train_m['loss']:.4f}")
        print(f"  Val   Loss : {val_m['loss']:.4f}")
        print(f"  Val   AUC  : {val_m['auc']:.4f}")
        print(f"  Time       : {elapsed:.0f}s")
        if CFG.DEVICE == "cuda":
            print(f"  VRAM       : {torch.cuda.memory_allocated()/1e9:.2f} GB "
                  f"(peak {torch.cuda.max_memory_allocated()/1e9:.2f} GB)")

        history.append({
            "epoch": epoch,
            "train_loss": train_m["loss"],
            "val_loss":   val_m["loss"],
            "val_auc":    val_m["auc"],
        })

        if val_m["auc"] > best_auc:
            best_auc   = val_m["auc"]
            no_improve = 0
            ckpt_path  = os.path.join(CFG.OUTPUT_DIR, f"best_fold{fold}.pt")
            torch.save({
                "epoch": epoch, "model": model.state_dict(),
                "val_auc": best_auc, "label2idx": label2idx,
            }, ckpt_path)
            print(f"  ✓ Saved → {ckpt_path}  (AUC={best_auc:.4f})")
        else:
            no_improve += 1
            print(f"  No improvement ({no_improve}/{patience})")

        if no_improve >= patience:
            print(f"\nEarly stopping. Best AUC: {best_auc:.4f}")
            break

    pd.DataFrame(history).to_csv(
        os.path.join(CFG.OUTPUT_DIR, f"history_fold{fold}.csv"), index=False
    )
    print(f"\nDone. Best val AUC: {best_auc:.4f}")
    return best_auc


best_auc = train(fold=CFG.FOLD)

In [ ]:
# ── Inference → submission.csv ────────────────────────────────────────────────
import glob as glob_module


@torch.no_grad()
def predict_soundscape(model, audio_path, label2idx, idx2label):
    model.eval()
    try:
        audio = load_audio(audio_path)
    except Exception as e:
        print(f"  Error: {e}")
        return pd.DataFrame()

    filename = os.path.splitext(os.path.basename(audio_path))[0]
    rows, offset = [], 0

    while offset + CFG.N_SAMPLES <= len(audio):
        chunk   = audio[offset: offset + CFG.N_SAMPLES]
        offset += CFG.N_SAMPLES
        end_sec = offset // CFG.SAMPLE_RATE

        mel = normalize_mel(audio_to_melspec(chunk))
        t   = torch.tensor(mel).unsqueeze(0).unsqueeze(0).to(CFG.DEVICE)

        preds = []
        for tta in [t, t.flip(-1)]:
            with autocast(enabled=CFG.USE_AMP):
                preds.append(torch.sigmoid(model(tta)).cpu().numpy()[0])
        probs = np.mean(preds, axis=0)

        row = {"row_id": f"{filename}_{end_sec}"}
        for idx, label in idx2label.items():
            row[label] = float(probs[idx])
        rows.append(row)

    return pd.DataFrame(rows)


def run_inference(fold=CFG.FOLD):
    label2idx, idx2label = build_label_map(CFG.TAXONOMY)

    ckpt_path = os.path.join(CFG.OUTPUT_DIR, f"best_fold{fold}.pt")
    model = BirdCLEFModel().to(CFG.DEVICE)
    ckpt  = torch.load(ckpt_path, map_location=CFG.DEVICE)
    model.load_state_dict(ckpt["model"])
    print(f"Loaded checkpoint (epoch {ckpt.get('epoch','?')}, "
          f"AUC={ckpt.get('val_auc',0):.4f})")

    sub_template  = pd.read_csv(CFG.SUB_CSV, nrows=1)
    expected_cols = list(sub_template.columns)

    sound_files = (
        glob_module.glob(os.path.join(CFG.TEST_SOUNDS, "*.ogg")) +
        glob_module.glob(os.path.join(CFG.TEST_SOUNDS, "*.wav"))
    )
    print(f"Found {len(sound_files)} test soundscape(s)")

    if not sound_files:
        print("No test soundscapes — saving dummy submission")
        sub = pd.read_csv(CFG.SUB_CSV)
        out = os.path.join(CFG.OUTPUT_DIR, "submission.csv")
        sub.to_csv(out, index=False)
        print(f"Saved → {out}")
        return

    all_rows = []
    for i, sf_path in enumerate(sorted(sound_files)):
        print(f"  [{i+1}/{len(sound_files)}] {os.path.basename(sf_path)}")
        df_pred = predict_soundscape(model, sf_path, label2idx, idx2label)
        if not df_pred.empty:
            all_rows.append(df_pred)

    submission = pd.concat(all_rows, ignore_index=True)
    for col in expected_cols:
        if col not in submission.columns:
            submission[col] = 1.0 / CFG.NUM_CLASSES
    submission = submission[expected_cols]

    out = os.path.join(CFG.OUTPUT_DIR, "submission.csv")
    submission.to_csv(out, index=False)
    print(f"\nSubmission saved → {out}")
    print(f"Shape: {submission.shape}")
    display(submission.head(3))


run_inference(fold=CFG.FOLD)

In [ ]:
# ── Training History Plot ─────────────────────────────────────────────────────
import matplotlib.pyplot as plt

hist = pd.read_csv(os.path.join(CFG.OUTPUT_DIR, f"history_fold{CFG.FOLD}.csv"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(hist["epoch"], hist["train_loss"], label="Train Loss")
axes[0].plot(hist["epoch"], hist["val_loss"],   label="Val Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Loss Curve"); axes[0].legend()

axes[1].plot(hist["epoch"], hist["val_auc"], color="green", label="Val AUC")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("AUC")
axes[1].set_title("Validation AUC"); axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(CFG.OUTPUT_DIR, "training_history.png"), dpi=120)
plt.show()
print(f"Best AUC: {hist['val_auc'].max():.4f} at epoch {hist['val_auc'].idxmax()+1}")